# 🎲 Yahtzee con el método de Montecarlo — versión mejorada

**Actividad didáctica 2-M1 · Simulación** · Johnatan Giraldo Cardona

Este notebook parte del notebook guía entregado por el profesor (`simulacion_01.ipynb`, que se conserva sin cambios en el repositorio) y sigue la metodología **USA → MODIFICA → CREA**:

| Etapa | Qué se hizo |
|---|---|
| **USA** | Se ejecutó el notebook guía tal como venía y se analizaron sus resultados. |
| **MODIFICA** | Se corrigieron 3 errores que invalidaban los resultados y se mejoró el rendimiento y la robustez. |
| **CREA** | Se agregaron experimentos nuevos para demostrar la **Ley de los Grandes Números**. |

Cada cambio está marcado en el código con un comentario `# [MODIFICA Mx]` o `# [CREA Cx]`. El detalle está en `MODIFICACIONES.md`.

**Modelo probabilístico:** cada dado es una variable aleatoria uniforme discreta, $P(X=k)=1/6$ para $k\in\{1,\dots,6\}$, con $E[X]=3.5$ y $Var(X)=35/12\approx2.917$.

## 1. Motor del juego y una partida completa (celda guía 1)

**USA:** motor del juego, estrategia Montecarlo con 32 máscaras de bloqueo y una partida narrada.

**MODIFICA:**
- **M1/M4** Las estadísticas solo cuentan los dados que realmente se lanzan. Antes también contaban los bloqueados y sesgaban el histograma de caras.
- **M5** `SIMS_PER_MASK` se lee al momento de decidir. Antes quedaba fijo en 300 desde que se definía la función, así que los escenarios de la celda 2 no cambiaban nada.
- **M6** Se corrige el conteo de lanzamientos de la simulación (antes `rolls_remaining - 1`). Con el error, en el último lanzamiento la IA relanzaba **los 5 dados** el 100 % de las veces y perdía lo que había conservado.
- **M2/M7** Rendimiento: caché de puntajes y evaluación una sola vez de las máscaras equivalentes.

In [9]:
import os
import random
import itertools
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from functools import lru_cache
from typing import List, Dict, Optional, Tuple

# ===============================
# Parámetros Montecarlo
# ===============================
SIMS_PER_MASK = 300  # nº de simulaciones por máscara de bloqueo (32 máx). Ajusta si quieres más/menos precisión.
SEED = 42            # fija semilla para reproducibilidad; pon None para aleatorio

if SEED is not None:
    random.seed(SEED)

# [CREA C0] Carpeta donde se guardan los gráficos generados
CARPETA_RESULTADOS = "resultados"
os.makedirs(CARPETA_RESULTADOS, exist_ok=True)

# ===============================
# Utilidades de dados
# ===============================
def roll_single_die() -> int:
    """Devuelve un entero uniforme en [1,6]."""
    return random.randint(1, 6)

def roll_dice(values: List[int], locked: List[bool]) -> List[int]:
    """Lanza dados no bloqueados, actualizando 'values' in place.
    [MODIFICA M1] Devuelve los valores realmente lanzados (para las estadísticas)."""
    thrown = []
    for i, is_locked in enumerate(locked):
        if not is_locked:
            values[i] = roll_single_die()
            thrown.append(values[i])
    return thrown

def has_sequence(values: List[int], needed_len: int) -> bool:
    """¿Existe una secuencia consecutiva de longitud 'needed_len'? (ignora duplicados)."""
    uniq = sorted(set(values))
    if not uniq:
        return False
    longest = curr = 1
    for i in range(1, len(uniq)):
        if uniq[i] == uniq[i-1] + 1:
            curr += 1
            longest = max(longest, curr)
        else:
            curr = 1
    return longest >= needed_len

# ===============================
# Puntuación
# ===============================
CATEGORIES_ORDER = [
    "ones","twos","threes","fours","fives","sixes",
    "threeOfKind","fourOfKind","fullHouse","smallStraight",
    "largeStraight","yahtzee","chance"
]

@lru_cache(maxsize=None)  # [MODIFICA M2] caché: solo existen 252 combinaciones distintas de 5 dados
def _score_sorted(category: str, dice: Tuple[int, ...]) -> int:
    counts = Counter(dice)
    freqs = list(counts.values())
    total = sum(dice)
    if category in ["ones","twos","threes","fours","fives","sixes"]:
        num = {"ones":1, "twos":2, "threes":3, "fours":4, "fives":5, "sixes":6}[category]
        return counts.get(num, 0) * num
    if category == "threeOfKind":
        return total if any(f >= 3 for f in freqs) else 0
    if category == "fourOfKind":
        return total if any(f >= 4 for f in freqs) else 0
    if category == "fullHouse":
        return 25 if (2 in freqs and 3 in freqs) else 0
    if category == "smallStraight":
        return 30 if has_sequence(dice, 4) else 0
    if category == "largeStraight":
        return 40 if has_sequence(dice, 5) else 0
    if category == "yahtzee":
        return 50 if 5 in freqs else 0
    if category == "chance":
        return total
    return 0

def score_hand(category: str, dice: List[int]) -> int:
    return _score_sorted(category, tuple(sorted(dice)))

def best_category_score(dice: List[int], available: List[str]) -> Tuple[str, int]:
    """Devuelve (mejor_categoria, puntuación) para los dados dados."""
    best_cat, best = None, -1
    for cat in available:
        sc = score_hand(cat, dice)
        if sc > best:
            best = sc
            best_cat = cat
    return best_cat, best

# ===============================
# Estructuras de juego
# ===============================
@dataclass
class PlayerState:
    name: str
    scores: Dict[str, Optional[int]] = field(default_factory=lambda: {c: None for c in CATEGORIES_ORDER})
    total: int = 0

    def available_categories(self) -> List[str]:
        return [c for c,v in self.scores.items() if v is None]

    def set_score(self, category: str, points: int) -> None:
        self.scores[category] = points
        self.total = sum(v for v in self.scores.values() if v is not None)

@dataclass
class GameStats:
    total_rolls: int = 0
    face_hist: Counter = field(default_factory=Counter)

    def update_from_roll(self, thrown: List[int]) -> None:
        # [MODIFICA M4] Solo se cuentan los dados lanzados (antes se sumaban los 5, incluidos los bloqueados)
        self.total_rolls += len(thrown)
        self.face_hist.update(thrown)

@dataclass
class GameState:
    players: List[PlayerState]
    current_player_idx: int = 0
    turn_in_round: int = 0  # 0..12 (13 turnos)
    dice_values: List[int] = field(default_factory=lambda: [0]*5)
    locked: List[bool] = field(default_factory=lambda: [False]*5)
    rolls_left: int = 3
    stats: GameStats = field(default_factory=GameStats)

    def reset_turn(self):
        self.dice_values = [0]*5
        self.locked = [False]*5
        self.rolls_left = 3

    @property
    def current_player(self) -> PlayerState:
        return self.players[self.current_player_idx]

    def advance_player(self):
        self.current_player_idx = (self.current_player_idx + 1) % len(self.players)
        if self.current_player_idx == 0:
            self.turn_in_round += 1

# ===============================
# Montecarlo para decidir bloqueos
# ===============================
def all_keep_masks() -> List[List[bool]]:
    """Genera las 32 máscaras posibles de bloqueo para 5 dados."""
    masks = []
    for m in range(32):  # 0..31
        mask = [(m >> i) & 1 == 1 for i in range(5)]
        masks.append(mask)
    return masks

def rollout_keep_mask(values: List[int]) -> List[bool]:
    """[MODIFICA M6] Regla simple para los lanzamientos posteriores dentro de la
    simulación: conservar el valor más repetido (en empate, el más alto)."""
    counts = Counter(values)
    target = max(counts, key=lambda v: (counts[v], v))
    return [v == target for v in values]

def simulate_expected_value_after_keeps(
    current_values: List[int],
    keep_mask: List[bool],
    rolls_remaining: int,
    available_categories: List[str],
    sims: int
) -> float:
    """
    Estima el valor esperado de la mejor categoría si:
    - conservas (keep_mask) algunos dados ahora y lanzas el resto,
    - y en los lanzamientos que queden después conservas el valor más repetido.
    [MODIFICA M6] 'rolls_remaining' incluye el lanzamiento que se va a hacer.
    """
    if rolls_remaining <= 0 or all(keep_mask):
        # Sin tiradas o plantarse: solo puntuar ahora
        _, sc = best_category_score(current_values, available_categories)
        return float(sc)

    total_score = 0.0
    for _ in range(sims):
        vals = current_values[:]
        roll_dice(vals, keep_mask)                    # próximo lanzamiento
        for _r in range(rolls_remaining - 1):         # lanzamientos posteriores
            roll_dice(vals, rollout_keep_mask(vals))
        _, sc = best_category_score(vals, available_categories)
        total_score += sc
    return total_score / sims if sims > 0 else 0.0

def choose_keep_mask_montecarlo(
    current_values: List[int],
    rolls_remaining: int,
    available_categories: List[str],
    sims_per_mask: Optional[int] = None
) -> List[bool]:
    """
    Elige la máscara de bloqueo que maximiza el valor esperado estimado al final del turno.
    """
    # [MODIFICA M5] Antes el valor por defecto quedaba fijo en 300 al definir la función
    if sims_per_mask is None:
        sims_per_mask = SIMS_PER_MASK
    best_mask = None
    best_ev = -1.0
    evaluated = set()
    for mask in all_keep_masks():
        # [MODIFICA M7] Máscaras que conservan los mismos valores son equivalentes
        key = tuple(sorted(v for v, keep in zip(current_values, mask) if keep))
        if key in evaluated:
            continue
        evaluated.add(key)
        # [MODIFICA M6] Antes se pasaba rolls_remaining - 1 (en el último lanzamiento simulaba 0 tiradas)
        ev = simulate_expected_value_after_keeps(
            current_values, mask, rolls_remaining, available_categories, sims_per_mask
        )
        if ev > best_ev:
            best_ev = ev
            best_mask = mask
    return best_mask

# ===============================
# Flujo del juego
# ===============================
def play_one_turn(state: GameState, verbose: bool = True):
    """Un turno completo para el jugador actual: hasta 3 tiradas + elección de categoría."""
    player = state.current_player
    state.reset_turn()

    # Tirada inicial
    thrown = roll_dice(state.dice_values, state.locked)
    state.stats.update_from_roll(thrown)
    state.rolls_left -= 1
    if verbose:
        print(f"\n{player.name} — Tirada 1:", state.dice_values)

    # Hasta 2 rerolls con estrategia MC
    while state.rolls_left > 0:
        keep_mask = choose_keep_mask_montecarlo(
            state.dice_values, state.rolls_left, player.available_categories()
        )

        # Si la máscara "keep_mask" implica mantener todo tal cual (keep 5), podemos decidir plantarnos:
        if all(keep_mask):
            if verbose:
                print(f"{player.name} decide plantarse con:", state.dice_values)
            break

        # Bloquear según la máscara
        state.locked = keep_mask[:]
        # Reroll
        thrown = roll_dice(state.dice_values, state.locked)
        state.stats.update_from_roll(thrown)
        state.rolls_left -= 1
        if verbose:
            tirada_num = 3 - state.rolls_left
            print(f"{player.name} — Tirada {tirada_num}:", state.dice_values, "| keep:", keep_mask)

    # Elegir mejor categoría disponible para la mano final
    cat, sc = best_category_score(state.dice_values, player.available_categories())
    player.set_score(cat, sc)
    if verbose:
        print(f"→ {player.name} elige categoría '{cat}' y anota {sc} puntos. Total = {player.total}")

def play_game(verbose_each_turn: bool = False, quiet: bool = False) -> GameState:
    players = [PlayerState("Jugador 1"), PlayerState("Jugador 2")]
    state = GameState(players=players)
    total_rounds = 13

    if not quiet:  # [MODIFICA M8] modo silencioso para simular muchas partidas
        print("🎲 Iniciando simulación de Montecarlo (Motor de IA tomando decisiones)...")
    for r in range(total_rounds):
        for _p in range(len(players)):
            play_one_turn(state, verbose=verbose_each_turn)
            state.advance_player()  # <--- ¡Esta es la línea que faltaba!
    if not quiet:
        print("✅ Simulación completada.\n")
    return state

# ===============================
# Ejecutar simulación completa
# ===============================
state = play_game(verbose_each_turn=True)

# ===============================
# Mostrar resultados
# ===============================
def print_scoreboard(state: GameState):
    print("\n\n===== MARCADOR FINAL =====")
    for p in state.players:
        print(f"\n{p.name}: {p.total} puntos")
        for cat in CATEGORIES_ORDER:
            v = p.scores[cat]
            print(f"  - {cat:13s}: {v if v is not None else 0}")
    # Ganador
    if state.players[0].total > state.players[1].total:
        winner = state.players[0].name
    elif state.players[1].total > state.players[0].total:
        winner = state.players[1].name
    else:
        winner = "Empate"
    print(f"\n🏆 Ganador: {winner}")

def print_stats(state: GameState):
    print("\n===== ESTADÍSTICAS =====")
    print(f"Total de lanzamientos (dados individuales): {state.stats.total_rolls}")
    total_faces = sum(state.stats.face_hist.values())
    for face in range(1,7):
        c = state.stats.face_hist.get(face, 0)
        p = (c/total_faces)*100 if total_faces else 0.0
        print(f" Cara {face}: {c} veces ({p:.2f}%)")

print_scoreboard(state)
print_stats(state)

🎲 Iniciando simulación de Montecarlo (Motor de IA tomando decisiones)...

Jugador 1 — Tirada 1: [6, 1, 1, 6, 3]
Jugador 1 — Tirada 2: [6, 5, 3, 6, 6] | keep: [True, False, False, True, False]
Jugador 1 — Tirada 3: [6, 5, 5, 6, 6] | keep: [True, True, False, True, True]
→ Jugador 1 elige categoría 'threeOfKind' y anota 28 puntos. Total = 28

Jugador 2 — Tirada 1: [4, 3, 3, 2, 1]
Jugador 2 decide plantarse con: [4, 3, 3, 2, 1]
→ Jugador 2 elige categoría 'smallStraight' y anota 30 puntos. Total = 30

Jugador 1 — Tirada 1: [3, 5, 4, 4, 3]
Jugador 1 — Tirada 2: [2, 5, 4, 3, 6] | keep: [False, True, True, False, False]
Jugador 1 decide plantarse con: [2, 5, 4, 3, 6]
→ Jugador 1 elige categoría 'largeStraight' y anota 40 puntos. Total = 68

Jugador 2 — Tirada 1: [2, 6, 6, 2, 3]
Jugador 2 — Tirada 2: [2, 6, 6, 6, 1] | keep: [True, True, True, False, False]
Jugador 2 — Tirada 3: [6, 6, 6, 6, 2] | keep: [False, True, True, True, False]
→ Jugador 2 elige categoría 'threeOfKind' y anota 26 puntos

## 2. Comparación de escenarios de `SIMS_PER_MASK` (celda guía 2)

**USA:** la guía comparaba 10, 1.000 y 100.000 simulaciones por máscara con **una sola partida** por escenario. Los tiempos salían iguales (~5 s) porque el parámetro no se aplicaba (error M5).

**MODIFICA:**
- **M9** Ya corregido M5, 100.000 simulaciones por máscara tardarían horas. Se usan 10, 100 y 1.000.
- **M9** Cada escenario juega **10 partidas** y se reporta el promedio con su intervalo de confianza: una sola partida no permite comparar nada, porque el azar domina.
- **M10** Se corrige la advertencia de seaborn (`palette` sin `hue`).

In [10]:
import time
import math
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# [MODIFICA M9] Escenarios ejecutables y varias partidas por escenario
escenarios_sims = [10, 100, 1000]
PARTIDAS_POR_ESCENARIO = 10
resultados_comparativa = []

for sims in escenarios_sims:
    print(f"\n⏳ Ejecutando {PARTIDAS_POR_ESCENARIO} partidas con SIMS_PER_MASK = {sims}...")
    SIMS_PER_MASK = sims          # ahora sí se aplica (MODIFICA M5)
    random.seed(SEED)

    puntos = []
    start_time = time.time()
    for _ in range(PARTIDAS_POR_ESCENARIO):
        estado_simulado = play_game(verbose_each_turn=False, quiet=True)
        puntos += [p.total for p in estado_simulado.players]
    elapsed_time = time.time() - start_time

    media = sum(puntos) / len(puntos)
    desv = (sum((x - media) ** 2 for x in puntos) / (len(puntos) - 1)) ** 0.5
    resultados_comparativa.append({
        "Sims por Máscara": sims,
        "Tiempo (s)": round(elapsed_time, 2),
        "Seg. por partida": round(elapsed_time / PARTIDAS_POR_ESCENARIO, 3),
        "Puntaje promedio": round(media, 1),
        "IC 95 % (±)": round(1.96 * desv / math.sqrt(len(puntos)), 1),
        "Mínimo": min(puntos),
        "Máximo": max(puntos),
    })

SIMS_PER_MASK = 300  # valor original de la guía para el resto del notebook

# Presentar resultados en una tabla
df_comparativo = pd.DataFrame(resultados_comparativa)
print("\n=== TABLA COMPARATIVA DE ESCENARIOS ===")
print(df_comparativo.to_string(index=False))

# Graficar el tiempo de ejecución y el puntaje según el número de simulaciones
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
sns.barplot(x="Sims por Máscara", y="Tiempo (s)", data=df_comparativo, hue="Sims por Máscara",
            palette="viridis", legend=False, ax=ax1)  # [MODIFICA M10]
ax1.set_title("Tiempo de Ejecución según SIMS_PER_MASK", fontsize=12, fontweight="bold")
ax1.set_ylabel(f"Tiempo total de {PARTIDAS_POR_ESCENARIO} partidas (s)")
ax1.set_xlabel("Simulaciones por Máscara")

# [CREA C1] Calidad de las decisiones: puntaje promedio con intervalo de confianza
ax2.errorbar(df_comparativo["Sims por Máscara"].astype(str), df_comparativo["Puntaje promedio"],
             yerr=df_comparativo["IC 95 % (±)"], marker="o", capsize=6)
ax2.set_title("Puntaje promedio por jugador (IC 95 %)", fontsize=12, fontweight="bold")
ax2.set_xlabel("Simulaciones por Máscara")
ax2.set_ylabel("Puntos")
plt.tight_layout()
plt.savefig(os.path.join(CARPETA_RESULTADOS, "escenarios_sims.png"), dpi=150, bbox_inches="tight")
plt.show()

ModuleNotFoundError: No module named 'matplotlib'

## 3. Diagrama de flujo y estudio de convergencia (celda guía 3)

**USA:** diagrama de flujo con Graphviz y convergencia del valor esperado (EV) de conservar los dos 3 en `[3, 3, 5, 2, 6]`.

**MODIFICA:**
- **M11** Se elimina una arista hacia un nodo inexistente (`mejor_mask_decision`), que creaba un nodo suelto en el diagrama.
- **M11** Si Graphviz no está instalado, el notebook avisa y continúa en lugar de detenerse.
- **M12** La guía tomaba como "valor verdadero" la última estimación, lo cual no demuestra convergencia.

**CREA:**
- **C2** Se calcula el **valor exacto** del EV enumerando todos los resultados posibles, para comparar contra un valor real.
- **C2** Se grafica el error contra la curva teórica $1/\sqrt{n}$.

In [ ]:
import numpy as np

# =========================================================
# 1. GENERACIÓN DEL DIAGRAMA DE FLUJO (Graphviz)
# =========================================================
def generar_diagrama_flujo():
    from graphviz import Digraph
    dot = Digraph(comment='Algoritmo Yahtzee Montecarlo', format='png')
    dot.attr(rankdir='TB', size='8,10', dpi='300')

    # Estilos de nodos
    dot.attr('node', shape='box', style='filled,rounded', color='black', fillcolor='#E3F2FD', fontname='Arial', fontsize='10')

    # Definir nodos
    dot.node('inicio', 'Inicio de Turno\n(Tirada Inicial: 5 Dados)', fillcolor='#C8E6C9', shape='ellipse')
    dot.node('eval_tiros', '¿Quedan tiros disponibles\n(rolls_left > 0)?', shape='diamond', fillcolor='#FFF9C4')
    dot.node('eval_mascaras', 'Evaluar las 32 Máscaras\nPosibles de Bloqueo (Keep Masks)', fillcolor='#E1BEE7')
    dot.node('montecarlo', 'Para cada máscara:\nSimular escenarios futuros (N lanzamientos)\ny calcular Valor Esperado (EV)', fillcolor='#BBDEFB')
    dot.node('mejor_mascara', 'Seleccionar máscara con\nmáximo Valor Esperado (EV)', fillcolor='#BBDEFB')
    dot.node('plantarse', '¿La mejor máscara es\nmantener todos los dados?', shape='diamond', fillcolor='#FFF9C4')
    dot.node('actualizar_dados', 'Lanzar dados no bloqueados\nDisminuir rolls_left', fillcolor='#FFE0B2')
    dot.node('anotar', 'Anotar en la mejor categoría\ndisponible en la hoja', fillcolor='#F8BBD0')
    dot.node('fin', 'Fin del Turno', fillcolor='#FFCDD2', shape='ellipse')

    # Conexiones
    dot.edge('inicio', 'eval_tiros')
    dot.edge('eval_tiros', 'eval_mascaras', label='Sí')
    dot.edge('eval_tiros', 'anotar', label='No')
    dot.edge('eval_mascaras', 'montecarlo')
    dot.edge('montecarlo', 'mejor_mascara')
    # [MODIFICA M11] Se eliminó dot.edge('mejor_mask_decision', 'plantarse'): ese nodo no existía
    dot.edge('mejor_mascara', 'plantarse')
    dot.edge('plantarse', 'anotar', label='Sí')
    dot.edge('plantarse', 'actualizar_dados', label='No')
    dot.edge('actualizar_dados', 'eval_tiros')
    dot.edge('anotar', 'fin')

    return dot

# [MODIFICA M11] Si Graphviz no está instalado, se informa y el notebook continúa
try:
    dot = generar_diagrama_flujo()
    dot.render(os.path.join(CARPETA_RESULTADOS, 'diagrama_flujo_yahtzee'), cleanup=True)
    print("✅ Diagrama de flujo generado: resultados/diagrama_flujo_yahtzee.png")
    from IPython.display import Image, display
    display(Image(os.path.join(CARPETA_RESULTADOS, 'diagrama_flujo_yahtzee.png'), width=420))
except Exception as e:
    print(f"⚠️ No se generó el diagrama (instala Graphviz: 'brew install graphviz' y 'pip install graphviz'): {e}")

# =========================================================
# 2. ESTUDIO DE CONVERGENCIA (Ley de los Grandes Números)
# =========================================================
def exact_expected_value_after_keeps(current_values, keep_mask, rolls_remaining, available_categories):
    """[CREA C2] Valor esperado EXACTO: enumera todos los resultados posibles (cada uno
    con probabilidad 1/6^k) siguiendo la misma regla que la simulación."""
    if rolls_remaining <= 0 or all(keep_mask):
        return float(best_category_score(current_values, available_categories)[1])
    unlocked = [i for i, keep in enumerate(keep_mask) if not keep]
    outcomes = list(itertools.product(range(1, 7), repeat=len(unlocked)))
    total = 0.0
    for out in outcomes:
        vals = list(current_values)
        for i, v in zip(unlocked, out):
            vals[i] = v
        total += exact_expected_value_after_keeps(vals, rollout_keep_mask(vals),
                                                  rolls_remaining - 1, available_categories)
    return total / len(outcomes)

def reporte_convergencia_lln():
    # Escenario de dados de prueba: Buscamos Full House o Trío con dados iniciales [3, 3, 5, 2, 6]
    dados_iniciales = [3, 3, 5, 2, 6]
    categorias_disponibles = ["threes", "fullHouse", "threeOfKind", "chance"]
    # Queremos evaluar mantener los dos 3s (Máscara: [True, True, False, False, False])
    mascara_keep = [True, True, False, False, False]

    valor_exacto = exact_expected_value_after_keeps(dados_iniciales, mascara_keep, 2, categorias_disponibles)
    print(f"\n🎯 Valor esperado EXACTO (enumeración de todos los casos): {valor_exacto:.4f}")

    valores_iteraciones = [10, 100, 500, 1000, 5000, 10000, 50000, 100000]
    ev_estimados = []
    tiempos = []

    random.seed(SEED)
    print("\n⏳ Iniciando experimento de convergencia...")
    for sims in valores_iteraciones:
        t_init = time.time()
        ev = simulate_expected_value_after_keeps(
            current_values=dados_iniciales,
            keep_mask=mascara_keep,
            rolls_remaining=2,
            available_categories=categorias_disponibles,
            sims=sims
        )
        t_elapsed = time.time() - t_init
        ev_estimados.append(ev)
        tiempos.append(t_elapsed)
        print(f"   Iteraciones: {sims:6d} | EV Estimado: {ev:.4f} | Error: {abs(ev - valor_exacto):.4f} | Tiempo: {t_elapsed:.4f}s")

    # Crear DataFrame de resultados
    df_conv = pd.DataFrame({
        "Iteraciones": valores_iteraciones,
        "EV Estimado": ev_estimados,
        "Error absoluto": [abs(e - valor_exacto) for e in ev_estimados],
        "Tiempo de Cálculo (s)": tiempos
    })

    # Graficar la Convergencia
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    # Subplot 1: Convergencia de EV (Ley de los Grandes Números)
    sns.lineplot(x="Iteraciones", y="EV Estimado", data=df_conv, ax=ax1, marker="o", color="b", linewidth=2)
    ax1.set_xscale('log')
    # [MODIFICA M12] La referencia es el valor exacto, no la última estimación
    ax1.axhline(y=valor_exacto, color='r', linestyle='--', label=f'Valor exacto ({valor_exacto:.3f})')
    ax1.set_title("Estudio de Convergencia del Valor Esperado (EV)\n(Ley de los Grandes Números)", fontsize=12, fontweight='bold')
    ax1.set_xlabel("Iteraciones (Escala Logarítmica)")
    ax1.set_ylabel("Valor Esperado (EV)")
    ax1.legend()
    ax1.grid(True, which="both", ls="--")

    # [CREA C2] Subplot 2: el error disminuye como 1/sqrt(n)
    ax2.loglog(df_conv["Iteraciones"], df_conv["Error absoluto"], "o-", color="g", label="Error |EV estimado − EV exacto|")
    n = np.array(valores_iteraciones, dtype=float)
    ref = df_conv["Error absoluto"].iloc[0] * np.sqrt(n[0] / n)
    ax2.loglog(n, ref, "k--", label="Tendencia teórica ∝ 1/√n")
    ax2.set_title("Error de la estimación vs Iteraciones", fontsize=12, fontweight='bold')
    ax2.set_xlabel("Iteraciones (Escala Logarítmica)")
    ax2.set_ylabel("Error absoluto (Escala Logarítmica)")
    ax2.legend()
    ax2.grid(True, which="both", ls="--")

    plt.tight_layout()
    plt.savefig(os.path.join(CARPETA_RESULTADOS, "convergencia_ev.png"), dpi=150, bbox_inches="tight")
    plt.show()
    return df_conv, valor_exacto

# Ejecutar el análisis
df_conv, valor_exacto = reporte_convergencia_lln()

## 4. Análisis didáctico de una partida (celda guía 5)

**USA:** tabla de puntajes por categoría, gráfico comparativo y distribución de caras frente a 1/6.

**MODIFICA:**
- **M13** La guía volvía a copiar todo el motor del juego (≈200 líneas duplicadas) antes del análisis. Ahora reutiliza las funciones de la sección 1, así una corrección no queda solo en una de las dos copias.
- **M4** Gracias a esta corrección, el histograma de caras refleja solo los dados realmente lanzados.

In [ ]:
random.seed(SEED)
# Ejecutamos en silencio para priorizar el análisis final
estado_final = play_game(verbose_each_turn=False)

# =========================================================
# REFACTORIZACIÓN DIDÁCTICA: ANÁLISIS Y VISUALIZACIÓN
# =========================================================
def analisis_didactico_resultados(state: GameState):
    """Presenta los resultados de la simulación de forma visual y pedagógica."""

    # Configuración visual de Seaborn
    sns.set_theme(style="whitegrid")

    # --- 1. RESUMEN DEL PARTIDO (TABLA VISUAL) ---
    print("="*50)
    print(" 📊 ANÁLISIS DE LA PARTIDA Y TOMA DE DECISIONES")
    print("="*50)

    # Determinamos al ganador
    p1, p2 = state.players
    if p1.total > p2.total:
        ganador = p1.name
    elif p2.total > p1.total:
        ganador = p2.name
    else:
        ganador = "Empate"

    print(f"\n🏆 Resultado de la Simulación: ¡El ganador es {ganador}!\n")

    # Construimos un DataFrame de Pandas para que los estudiantes vean una tabla estructurada
    df_scores = pd.DataFrame({
        "Categoría": CATEGORIES_ORDER,
        p1.name: [p1.scores[c] if p1.scores[c] is not None else 0 for c in CATEGORIES_ORDER],
        p2.name: [p2.scores[c] if p2.scores[c] is not None else 0 for c in CATEGORIES_ORDER]
    })

    # Mostramos la tabla formateada
    print(df_scores.to_string(index=False))
    print("-" * 30)
    print(f"PUNTUACIÓN TOTAL | {p1.name}: {p1.total} | {p2.name}: {p2.total}")

    # --- GRÁFICO 1: COMPARACIÓN DE PUNTUACIONES ---
    plt.figure(figsize=(10, 5))
    ax = sns.barplot(x="Categoría", y="Valor", hue="Jugador",
                     data=pd.melt(df_scores, id_vars="Categoría", var_name="Jugador", value_name="Valor"),
                     palette="muted")
    plt.title("Comparación de Puntuaciones por Categoría", fontsize=14, fontweight="bold")
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Puntos Obtenidos")
    plt.xlabel("Categoría de Yahtzee")
    plt.tight_layout()
    plt.savefig(os.path.join(CARPETA_RESULTADOS, "puntuaciones_partida.png"), dpi=150, bbox_inches="tight")
    plt.show()

    # --- 2. ANÁLISIS DE PROBABILIDAD Y LA LEY DE LOS GRANDES NÚMEROS ---
    print("\n" + "="*50)
    print(" 🎲 ANÁLISIS DEL MOTOR DE DADOS (Validación de Montecarlo)")
    print("="*50)
    print("\nEn una simulación de Montecarlo, es vital comprobar si nuestro generador aleatorio es justo.")
    print(f"Teóricamente, cada cara de un dado perfecto tiene una probabilidad de: 1/6 = 16.67%")
    print(f"Veamos cómo se comportaron nuestros dados tras {state.stats.total_rolls} lanzamientos individuales:\n")

    total_faces = sum(state.stats.face_hist.values())
    caras = list(range(1, 7))
    frecuencias = [state.stats.face_hist.get(face, 0) for face in caras]
    probabilidades_empiricas = [(frec / total_faces) * 100 if total_faces else 0 for frec in frecuencias]

    # Imprimimos los resultados teóricos vs simulados
    for face, frec, prob in zip(caras, frecuencias, probabilidades_empiricas):
        error_marginal = abs(prob - 16.67)
        print(f" Cara {face}: Salió {frec} veces -> {prob:.2f}% (Diferencia vs Teoría: {error_marginal:.2f}%)")

    # --- GRÁFICO 2: DISTRIBUCIÓN DE PROBABILIDAD ---
    plt.figure(figsize=(8, 5))

    # Graficamos los datos empíricos
    bar_plot = sns.barplot(x=caras, y=probabilidades_empiricas, color="skyblue", label="Probabilidad Simulada")

    # Agregamos la línea teórica del 16.67%
    plt.axhline(y=16.67, color='red', linestyle='--', linewidth=2, label='Probabilidad Teórica (16.67%)')

    plt.title(f"Distribución de Caras tras {state.stats.total_rolls} lanzamientos", fontsize=14, fontweight="bold")
    plt.xlabel("Cara del Dado")
    plt.ylabel("Frecuencia Relativa (%)")
    plt.ylim(0, max(probabilidades_empiricas) + 5) # Damos espacio arriba
    plt.legend()

    # Añadir los porcentajes encima de las barras para máxima claridad
    for p in bar_plot.patches:
        bar_plot.annotate(format(p.get_height(), '.1f') + '%',
                          (p.get_x() + p.get_width() / 2., p.get_height()),
                          ha = 'center', va = 'center',
                          xytext = (0, 9),
                          textcoords = 'offset points')

    plt.savefig(os.path.join(CARPETA_RESULTADOS, "distribucion_caras_partida.png"), dpi=150, bbox_inches="tight")
    plt.show()

    print("\n💡 REFLEXIÓN DIDÁCTICA:")
    print("Observa el gráfico de distribución. La línea punteada roja representa la probabilidad matemática ideal.")
    print("Las barras azules son el resultado de la simulación. Con pocos cientos de dados todavía hay")
    print("desviaciones visibles; la sección 5 muestra qué pasa cuando el número de lanzamientos crece.")

# Llamamos a la nueva función de presentación
analisis_didactico_resultados(estado_final)

## 5. 🆕 Ley de los Grandes Números en los dados

**CREA C3.** En una partida se lanzan unos 250 dados, y con tan pocos las frecuencias se alejan bastante de 1/6 (sección 4). Aquí se lanzan **1.000.000 de dados** y se registra la evolución:

- de la frecuencia relativa de cada cara, que debe acercarse a 1/6;
- del promedio, que debe acercarse a $E[X]=3.5$;
- del error máximo, que debe disminuir aproximadamente como $1/\sqrt{n}$.

Al final se aplica la prueba chi-cuadrado de bondad de ajuste a la distribución uniforme. Con un nivel de significancia del 5 %, incluso un generador perfecto "rechaza" en 1 de cada 20 muestras. Por eso, además de la prueba sobre el millón de dados, se repite sobre **200 muestras independientes** y se verifica que la proporción de rechazos esté cerca del 5 % esperado.

In [ ]:
# [CREA C3] Evolución de las frecuencias y del promedio al aumentar los lanzamientos
random.seed(SEED)
N_DADOS = 1_000_000
puntos_control = set(np.unique(np.logspace(1, 6, 51).astype(int)))
conteo = [0] * 7
suma = 0
filas = []
for n in range(1, N_DADOS + 1):
    v = roll_single_die()
    conteo[v] += 1
    suma += v
    if n in puntos_control:
        frec = [conteo[k] / n for k in range(1, 7)]
        filas.append({"n": n, **{f"Cara {k}": frec[k - 1] for k in range(1, 7)},
                      "Promedio": suma / n,
                      "Error máximo": max(abs(f - 1/6) for f in frec)})
df_lln = pd.DataFrame(filas)

fig, axs = plt.subplots(1, 3, figsize=(17, 4.6))
for k in range(1, 7):
    axs[0].plot(df_lln["n"], df_lln[f"Cara {k}"] * 100, label=f"Cara {k}")
axs[0].axhline(100/6, color="k", linestyle="--", linewidth=2, label="1/6 = 16.67 %")
axs[0].set_xscale("log"); axs[0].set_title("Frecuencia relativa de cada cara", fontweight="bold")
axs[0].set_xlabel("Dados lanzados (log)"); axs[0].set_ylabel("%"); axs[0].legend(fontsize=8, ncol=2)

axs[1].plot(df_lln["n"], df_lln["Promedio"], color="purple")
axs[1].axhline(3.5, color="k", linestyle="--", label="E[X] = 3.5")
axs[1].set_xscale("log"); axs[1].set_title("Promedio de los dados", fontweight="bold")
axs[1].set_xlabel("Dados lanzados (log)"); axs[1].legend()

axs[2].loglog(df_lln["n"], df_lln["Error máximo"] * 100, "o-", markersize=3, label="Error máximo |frec − 1/6|")
nn = df_lln["n"].to_numpy(dtype=float)
axs[2].loglog(nn, 100 * np.sqrt((1/6) * (5/6) / nn), "k--", label="Desviación teórica √(p(1−p)/n)")
axs[2].set_title("El error disminuye como 1/√n", fontweight="bold")
axs[2].set_xlabel("Dados lanzados (log)"); axs[2].set_ylabel("Error (puntos %)"); axs[2].legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(CARPETA_RESULTADOS, "lln_dados.png"), dpi=150, bbox_inches="tight")
plt.show()

# Tabla resumen en algunos tamaños de muestra
resumen = df_lln[df_lln["n"].isin([10, 100, 1000, 10000, 100000, 1000000])][["n", "Promedio", "Error máximo"]].copy()
resumen["Error máximo"] = (resumen["Error máximo"] * 100).round(3).astype(str) + " pts %"
resumen["Promedio"] = resumen["Promedio"].round(4)
print(resumen.to_string(index=False))

# Prueba chi-cuadrado (gl = 5, alfa = 0.05 -> valor crítico 11.07)
def chi2_uniforme(cuentas):
    n = sum(cuentas)
    e = n / 6
    return sum((c - e) ** 2 / e for c in cuentas)

def valor_p_chi2_gl5(x):
    """P(Chi2(5) > x), fórmula cerrada para 5 grados de libertad."""
    return math.erfc(math.sqrt(x / 2)) + math.sqrt(2 * x / math.pi) * math.exp(-x / 2) * (1 + x / 3)

chi2 = chi2_uniforme(conteo[1:])
print(f"\nChi-cuadrado (1.000.000 de dados) = {chi2:.3f}  ·  valor p = {valor_p_chi2_gl5(chi2):.3f}  "
      f"(valor crítico 11.070) -> {'NO se rechaza' if chi2 < 11.070 else 'Se rechaza'} la uniformidad")

# Repetición de la prueba en 200 muestras independientes de 5.000 dados
MUESTRAS, TAM = 200, 5_000
rechazos = 0
for _ in range(MUESTRAS):
    c = [0] * 6
    for _ in range(TAM):
        c[roll_single_die() - 1] += 1
    rechazos += chi2_uniforme(c) > 11.070
print(f"Muestras que rechazan la uniformidad: {rechazos} de {MUESTRAS} ({rechazos / MUESTRAS * 100:.1f} %) "
      f"-> lo esperado para un generador uniforme es cerca del 5 %")

## 6. 🆕 Probabilidades de las combinaciones: Montecarlo vs teoría

**CREA C4.** Se simulan 100.000 lanzamientos de 5 dados y se compara la frecuencia de cada combinación con su probabilidad exacta. La probabilidad exacta se obtiene enumerando los $6^5 = 7776$ resultados equiprobables, y coincide con la tabla de la guía del laboratorio (Yahtzee 0,077 %, Póker 1,93 %, Full House 3,86 %, Escalera Mayor 3,09 %, Escalera Menor 12,35 %).

In [ ]:
# [CREA C4] Clasificación excluyente de un lanzamiento
def clasificar(d):
    f = sorted(Counter(d).values(), reverse=True)
    if f[0] == 5: return "Yahtzee"
    if f[0] == 4: return "Póker"
    if f == [3, 2]: return "Full House"
    if has_sequence(d, 5): return "Escalera Mayor"
    if has_sequence(d, 4): return "Escalera Menor"
    if f[0] == 3: return "Trío"
    if f[:2] == [2, 2]: return "Doble par"
    if f[0] == 2: return "Un par"
    return "Nada"

ORDEN = ["Yahtzee", "Póker", "Full House", "Escalera Mayor", "Escalera Menor", "Trío", "Doble par", "Un par", "Nada"]
exactas = Counter(clasificar(d) for d in itertools.product(range(1, 7), repeat=5))

random.seed(SEED)
N_LANZ = 100_000
simuladas = Counter(clasificar([roll_single_die() for _ in range(5)]) for _ in range(N_LANZ))

df_comb = pd.DataFrame({
    "Combinación": ORDEN,
    "Simulada (%)": [simuladas[k] / N_LANZ * 100 for k in ORDEN],
    "Teórica (%)": [exactas[k] / 7776 * 100 for k in ORDEN],
})
df_comb["Error abs."] = (df_comb["Simulada (%)"] - df_comb["Teórica (%)"]).abs()
df_comb["IC 95 % (±)"] = [1.96 * math.sqrt(p/100 * (1 - p/100) / N_LANZ) * 100 for p in df_comb["Simulada (%)"]]
print(df_comb.round(3).to_string(index=False))
dentro = (df_comb["Error abs."] <= df_comb["IC 95 % (±)"]).sum()
print(f"\n{dentro} de {len(ORDEN)} combinaciones tienen el valor teórico dentro del IC 95 % de la simulación.")

ax = df_comb.set_index("Combinación")[["Simulada (%)", "Teórica (%)"]].plot.bar(figsize=(11, 4.5), rot=30)
ax.set_title(f"Combinaciones en un lanzamiento de 5 dados (n = {N_LANZ:,})", fontweight="bold")
ax.set_ylabel("Probabilidad (%)")
plt.tight_layout()
plt.savefig(os.path.join(CARPETA_RESULTADOS, "probabilidades_combinaciones.png"), dpi=150, bbox_inches="tight")
plt.show()

## 7. 🆕 Montecarlo de partidas completas

**CREA C5.** Una sola partida no dice nada sobre el comportamiento del juego. Aquí se simulan **200 partidas** entre los dos jugadores, que usan la misma estrategia Montecarlo, y se observa cómo se estabilizan:

- el **puntaje promedio**;
- el **porcentaje de victorias** del Jugador 1, que debe tender a cerca del 50 % porque ambos juegan igual.

Para que el experimento sea ágil se usan 100 simulaciones por máscara; en la sección 2, 100 y 1.000 simulaciones dieron promedios muy parecidos.

In [ ]:
# [CREA C5] Simulación de muchas partidas
SIMS_PER_MASK = 100
N_PARTIDAS = 200
random.seed(SEED)
registros = []
t0 = time.time()
for k in range(1, N_PARTIDAS + 1):
    est = play_game(verbose_each_turn=False, quiet=True)
    p1, p2 = est.players
    registros.append({"partida": k, "J1": p1.total, "J2": p2.total,
                      "gana_J1": 1 if p1.total > p2.total else 0,
                      "empate": 1 if p1.total == p2.total else 0,
                      "yahtzees": (p1.scores["yahtzee"] == 50) + (p2.scores["yahtzee"] == 50)})
print(f"⏱️ {N_PARTIDAS} partidas en {time.time() - t0:.1f} s")
df_part = pd.DataFrame(registros)
df_part.to_csv(os.path.join(CARPETA_RESULTADOS, "partidas.csv"), index=False)

puntajes = pd.concat([df_part["J1"], df_part["J2"]])
media, desv = puntajes.mean(), puntajes.std()
print(f"Puntaje promedio por jugador: {media:.1f} ± {1.96 * desv / math.sqrt(len(puntajes)):.1f} (IC 95 %)  ·  desviación {desv:.1f}")
print(f"Victorias J1: {df_part['gana_J1'].mean()*100:.1f} %  ·  J2: {(1 - df_part['gana_J1'] - df_part['empate']).mean()*100:.1f} %  ·  Empates: {df_part['empate'].sum()}")
print(f"Partidas con al menos un Yahtzee: {(df_part['yahtzees'] > 0).mean()*100:.1f} %")

# Evolución acumulada (Ley de los Grandes Números aplicada al juego)
n = df_part["partida"]
media_acum = ((df_part["J1"] + df_part["J2"]) / 2).cumsum() / n
victorias_acum = df_part["gana_J1"].cumsum() / n * 100

fig, axs = plt.subplots(1, 3, figsize=(17, 4.6))
axs[0].hist([df_part["J1"], df_part["J2"]], bins=15, label=["Jugador 1", "Jugador 2"], alpha=0.75)
axs[0].axvline(media, color="k", linestyle="--", label=f"Media {media:.1f}")
axs[0].set_title("Distribución de puntajes finales", fontweight="bold")
axs[0].set_xlabel("Puntos"); axs[0].set_ylabel("Partidas"); axs[0].legend()

axs[1].plot(n, media_acum, color="purple")
axs[1].axhline(media, color="k", linestyle="--", label=f"Media final {media:.1f}")
axs[1].set_title("Puntaje promedio acumulado", fontweight="bold")
axs[1].set_xlabel("Número de partidas"); axs[1].set_ylabel("Puntos"); axs[1].legend()

axs[2].plot(n, victorias_acum, color="teal")
axs[2].axhline(50, color="k", linestyle="--", label="50 % (misma estrategia)")
axs[2].set_ylim(0, 100)
axs[2].set_title("% de victorias acumulado del Jugador 1", fontweight="bold")
axs[2].set_xlabel("Número de partidas"); axs[2].set_ylabel("%"); axs[2].legend()
plt.tight_layout()
plt.savefig(os.path.join(CARPETA_RESULTADOS, "montecarlo_partidas.png"), dpi=150, bbox_inches="tight")
plt.show()

for corte in [10, 50, 100, 200]:
    print(f"  Tras {corte:3d} partidas -> promedio {media_acum.iloc[corte-1]:.1f} pts, victorias J1 {victorias_acum.iloc[corte-1]:.1f} %")
SIMS_PER_MASK = 300

## 8. Conclusiones

- **USA:** el notebook guía implementa bien el juego y la idea de usar Montecarlo para decidir qué dados conservar. Sin embargo, tres errores invalidaban los resultados: el parámetro de simulaciones no se aplicaba, la IA relanzaba todos los dados en el último lanzamiento y el histograma contaba dados que no se lanzaban.
- **MODIFICA:** con las correcciones, el escenario de simulaciones por máscara sí cambia el tiempo y la calidad de las decisiones, y la distribución de caras refleja solo los dados lanzados.
- **CREA:** los experimentos nuevos muestran la **Ley de los Grandes Números**:
  - las frecuencias de las caras convergen a 1/6 y el promedio a 3,5;
  - el error disminuye como $1/\sqrt{n}$;
  - las probabilidades de cada combinación coinciden con las exactas;
  - el EV estimado converge al EV exacto;
  - en las partidas completas, el promedio y el porcentaje de victorias se estabilizan a medida que crece el número de partidas.

Los valores numéricos de cada conclusión están en las salidas de las celdas anteriores y en el documento técnico.